In [0]:

query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY zip_prefix) AS geo_key,
    zip_prefix AS zip_code, 
    geolocation_city AS city,
    UPPER(geolocation_state) AS state
FROM (
    SELECT 
        zip_prefix,
        geolocation_city,
        geolocation_state,
        ROW_NUMBER() OVER (
            PARTITION BY zip_prefix
            ORDER BY geolocation_city
        ) AS rank
    FROM workspace.silver.geolocation
)
WHERE rank = 1
"""
df = spark.sql(query)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_geography")
)

In [0]:
%sql
-- Final verification: Checking for unique zip codes
SELECT 
    *
FROM workspace.gold.dim_geography 
LIMIT 10